# 4. Milestone: baseline error analysis

Use the baseline outputs from notebook 3. No improvements, tuning or additional training are needed here. Start with RF, fixed before seeing results.

Inspect per-class recall, confused activities, difficult participants and mixed windows. These observations will motivate the later experiments.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), Path.cwd() / 'harth_project', Path.cwd().parent]
            if (p / 'har_utils.py').exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from har_utils import *

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
required = ['RF_loso_predictions.csv', 'RF_loso_folds.csv']
if any(not (ROOT / 'results' / name).exists() for name in required):
    raise FileNotFoundError('Run the RF training cell in notebook 03 first, or copy its results files into this project.')
pred = pd.read_csv(ROOT / 'results/RF_loso_predictions.csv')
report = classification_report(pred.label, pred.predicted, labels=LABELS,
    target_names=list(ACTIVITIES.values()), output_dict=True, zero_division=0)
display(pd.DataFrame(report).T.round(3))

The pooled class report weights participants by window count. The baseline comparison instead averages subject-fold scores. Keep the two summaries distinct.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(pred.label, pred.predicted, labels=LABELS,
    display_labels=list(ACTIVITIES.values()), normalize='true', xticks_rotation=90,
    ax=ax, colorbar=False, values_format='.2f')
plt.tight_layout(); plt.show()

In [ ]:
folds = pd.read_csv(ROOT / 'results/RF_loso_folds.csv')
display(folds[['test_subjects', 'accuracy', 'macro_f1']].sort_values('macro_f1'))
errors = pred[pred.label != pred.predicted].copy()
confusions = errors.groupby(['label', 'predicted']).size().sort_values(ascending=False).head(12)
confusions.index = pd.MultiIndex.from_tuples(
    [(ACTIVITIES[a], ACTIVITIES[b]) for a,b in confusions.index], names=['true', 'predicted'])
display(confusions.to_frame('wrong windows'))

In [ ]:
pred['correct'] = pred.label == pred.predicted
pred['window_type'] = np.where(pred.purity == 1, 'pure', 'mixed')
display(pred.groupby('window_type').agg(windows=('correct','size'), accuracy=('correct','mean')))
display(errors[['subject','start_row','start_time','label','predicted','purity']].head(20))

## Stop here for the milestone

Record which classes and participants are difficult and whether mixed windows fail more often. No improvement is claimed before results exist. Notebook 5 contains feature experiments; notebook 6 contains tuning. Both are optional later work.